#### Inference with all models + Data Aug

In [ ]:
import numpy as np
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import torch
from PIL import Image
from transformers import ViTForImageClassification, ViTImageProcessor
from models.resnets.models import ResNet18, ResNet50
from torchvision import transforms

import os

with np.load("../data/bloodmnist.npz", allow_pickle=False) as dataset:
    t_images = dataset["train_images"]
    t_labels = dataset["train_labels"].reshape(-1)
    test_images = dataset["test_images"]
    test_labels = dataset["test_labels"].reshape(-1)

In [18]:
def batch_inference(model, processor=None, images=None, batch_size=32, device="cuda", data_aug=None): # batched inference to avoid out-of-memory
    all_probs = []
    
    for i in range(0, len(images), batch_size):
        batch_imgs = images[i : i + batch_size]

        transform = None
        if data_aug:
            if "rotation" == data_aug:
                #transform = transforms.RandomRot(degrees=90)
                batch_imgs = np.stack([
                    np.rot90(img, k=np.random.randint(4)).copy()
                    for img in batch_imgs
                ])
            elif "croping"  == data_aug:
                data_dim = batch_imgs[0].shape[:2]
                transform = transforms.RandomResizedCrop(size=data_dim, scale=(0.8, 1.0), ratio=(0.9, 1.1))
            elif data_aug:
                raise ValueError(f"Augmentation {data_aug} not known.")
        if transform:
            batch_imgs=[np.array(transform(Image.fromarray(img)))for img in batch_imgs]

        if processor!=None:
            inputs = processor(images=batch_imgs, return_tensors="pt").to(device)
        else:
            batch_imgs = np.stack(batch_imgs)
            inputs = torch.from_numpy(batch_imgs.transpose(0, 3, 1, 2).copy()).float()
            inputs = (inputs / 255.0 - 0.5) / 0.5
            inputs = inputs.to(device)
        
        with torch.inference_mode(), torch.amp.autocast(device_type=device, dtype=torch.float16):
            if processor != None:
                outputs = model(**inputs)
                logits = outputs.logits
            else:
                logits = model(inputs)
            probs = logits.softmax(dim=1).cpu().numpy()
            all_probs.append(probs)
        
        torch.cuda.empty_cache()
        
    return np.concatenate(all_probs, axis=0)

In [19]:



torch.set_num_threads(4)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

#model_path = Path("../models/vit")
#model_path = Path("../weights_bloodmnist/resnet18_28_1.pth")
model_path = Path("../weights_bloodmnist/resnet50_28_1.pth")
if not model_path.exists():
    raise FileNotFoundError(f"Model directory not found at: {model_path}")

print("Loading model (and processor)...")
processor=None
if "vit" in model_path.name.lower():
    processor = ViTImageProcessor.from_pretrained(model_path, local_files_only=True)
    model = ViTForImageClassification.from_pretrained(model_path, local_files_only=True)
if "resnet18" in model_path.name.lower():
    model = ResNet18(in_channels=3, num_classes=8)
    checkpoint = torch.load(model_path, map_location="cpu", weights_only=True)
    model.load_state_dict(checkpoint["net"], strict=True)
if "resnet50" in model_path.name.lower():
    model = ResNet50(in_channels=3, num_classes=8)
    checkpoint = torch.load(model_path, map_location="cpu", weights_only=True)
    model.load_state_dict(checkpoint["net"], strict=True)

model.to(device)
model.eval()

BATCH_SIZE = 32

#print("Process train set...")
#t_probs = batch_inference(model, processor, t_images, batch_size=BATCH_SIZE, device=device)

print("Process test set...")
test_probs = batch_inference(model, processor, test_images, batch_size=BATCH_SIZE, device=device, data_aug="rotation")

print("Save results...")
#t_df = pd.DataFrame(t_probs)
test_df = pd.DataFrame(test_probs)

#t_df.to_csv("../predictions/vit_bloodmnist_train.csv", index=True, header=False)
test_df.to_csv("../predictions/resnet50_bloodmnist_test_rot.csv", index=True, header=False)
print("Done!")

cuda
Loading model (and processor)...
Process test set...
Save results...
Done!
